# Analysis of PAC comodulogram metrics 


In [10]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency
from statsmodels.genmod.generalized_estimating_equations import GEE
from statsmodels.genmod.families import Binomial
import rpy2.robjects as robjects
from rpy2.robjects import pandas2ri
from rpy2.robjects.packages import importr 
from matplotlib.lines import Line2D

from figure_style import apply_style, set_panel_title, WT, NLGF
GENO_COLORS = {"NLGF": NLGF, "WT": WT}
GENO_ORDER = ["WT", "NLGF"]
apply_style()

In [11]:
# ─── Load data ────────────────────────────────────────────────────────────────
basepath = Path('/Users/loukia/Library/CloudStorage/Dropbox-UCL/Loukia Katsouri/DataProtocolsEquipment/Ephys_Analysis/RobinData/Analysis/LFP_analysis')
LFP_CSV_PATH = basepath / 'concatenated_lfp_stats.csv' 
OUTPUT_DIR = LFP_CSV_PATH.parent.parent / 'LFP_analysis'
os.makedirs(OUTPUT_DIR, exist_ok=True)


df = pd.read_csv(LFP_CSV_PATH)

# Clean names / categorical order
df["Genotype"] = pd.Categorical(df["Genotype"], categories=GENO_ORDER, ordered=True)
df["environment"] = df["environment"].astype(str)
df["mouse_name"] = df["mouse_name"].astype(str)
df["Experimenter"] = df["Experimenter"].astype(str)


## Step 1 — Normality (Shapiro–Wilk) and Step 2 — Levene's test, per metric × environment

In [ ]:
# ─── Metrics to analyse ───────────────────────────────────────────────────────
metrics = {
    "max_theta_freq": "Theta peak frequency (Hz)",
    "comodulogram_PAC_slow_gamma": "Theta–slow gamma PAC",
    "comodulogram_PAC_fast_gamma": "Theta–fast gamma PAC",
    "n_oscillatory_epochs_slow_gamm": "Slow gamma event count",
    "n_oscillatory_epochs_fast_gamma": "Fast gamma event count",
    "theta_running_slope": "Theta–running slope",
    "theta_running_intercept": "Theta–running intercept",
    "theta_running_rvalue": "Theta–running r",
    "theta_running_stderr": "Theta–running standard error",
    "theta_running_intercept_stderr": "Theta–running intercept standard error",
    "theta_gamma_modulation_index_fast_gamma": "Theta–Fast gamma modulation index",
    "theta_gamma_modulation_index_slow_gamma": "Theta–Slow gamma modulation index"
}

metrics = {k: v for k, v in metrics.items() if k in df.columns}

# ─── Step 1: Normality (Shapiro-Wilk) + Step 2: Levene's test, per metric × environment ──
# Tested on per-animal means (one value per mouse per environment), so a mouse
# contributing more than one session row doesn't pseudoreplicate the test — see
# Section 5 of CLAUDE.md. Both genotype groups must pass Shapiro (p > 0.05, n >= 3)
# for the metric × environment combination to be treated as normal. Levene's center
# is 'mean' (classic Levene) when normal, 'median' (Brown-Forsythe, robust to
# non-normality) otherwise. Both decisions drive the primary test choice in the
# next cell.
from scipy.stats import shapiro, levene

animal_agg_store  = {}   # (metric, environment) -> per-animal mean/median dataframe
normality_results = {}   # (metric, environment) -> bool (True = both genotypes normal)
levene_results    = {}   # (metric, environment) -> dict(statistic, p_value, center, equal_var)
diag_rows = []

for metric in metrics:
    for env in sorted(df["environment"].dropna().astype(str).unique()):
        sub = df.loc[df["environment"].astype(str) == env]
        if sub.empty:
            continue

        animal_df = (
            sub.groupby(["mouse_name", "Genotype"], observed=True)[metric]
            .agg(metric_mean="mean", metric_median="median", n_sessions="count")
            .reset_index()
        )
        animal_agg_store[(metric, env)] = animal_df

        shapiro_p = {}
        is_normal = True
        for geno in GENO_ORDER:
            vals = animal_df.loc[animal_df["Genotype"].astype(str) == geno, "metric_mean"].dropna().values
            if len(vals) < 3:
                is_normal = False
                shapiro_p[geno] = np.nan
            else:
                _, sw_p = shapiro(vals)
                shapiro_p[geno] = sw_p
                if sw_p <= 0.05:
                    is_normal = False
        normality_results[(metric, env)] = is_normal

        wt_vals   = animal_df.loc[animal_df["Genotype"].astype(str) == "WT",   "metric_mean"].dropna().values
        nlgf_vals = animal_df.loc[animal_df["Genotype"].astype(str) == "NLGF", "metric_mean"].dropna().values

        center = "mean" if is_normal else "median"
        if len(wt_vals) >= 2 and len(nlgf_vals) >= 2:
            lev_stat, lev_p = levene(wt_vals, nlgf_vals, center=center)
            equal_var = lev_p > 0.05
        else:
            lev_stat, lev_p, equal_var = np.nan, np.nan, np.nan
        levene_results[(metric, env)] = {
            "statistic": lev_stat, "p_value": lev_p,
            "center": center, "equal_var": equal_var,
        }

        diag_rows.append({
            "metric": metric, "environment": env,
            "n_WT_mice": len(wt_vals), "n_NLGF_mice": len(nlgf_vals),
            "shapiro_p_WT": shapiro_p.get("WT", np.nan),
            "shapiro_p_NLGF": shapiro_p.get("NLGF", np.nan),
            "normal": is_normal,
            "levene_center": center, "levene_stat": lev_stat, "levene_p": lev_p,
            "equal_var": equal_var,
        })

diagnostics_df = pd.DataFrame(diag_rows)
print(diagnostics_df.to_string(index=False))

## Step 3 — Primary statistical test (Student's/Welch's t-test if normal, Mann-Whitney U otherwise)

In [ ]:
"""
Primary statistical test per metric × environment, chosen from the previous
cell's Shapiro (normality) and Levene's (variance-homogeneity) results:
  - normal combination  -> t-test on per-animal means (Student's if Levene's
    equal_var is True, Welch's otherwise)
  - non-normal combination -> Mann-Whitney U on per-animal medians
Saves:
  lfp_stats_outputs/
    primary_test_results.csv
    primary_test_results.html
"""

import os
import numpy as np
import pandas as pd
from scipy.stats import ttest_ind, mannwhitneyu
from IPython.display import display, HTML


def _pstars(p):
    if pd.isna(p): return ""
    if p < 0.001:  return "***"
    if p < 0.01:   return "**"
    if p < 0.05:   return "*"
    return "ns"


test_results = []

for metric in metrics:
    for env in sorted(df["environment"].dropna().astype(str).unique()):
        key = (metric, env)
        if key not in animal_agg_store:
            continue

        animal_df = animal_agg_store[key]
        is_normal = normality_results[key]

        if is_normal:
            value_col = "metric_mean"
            equal_var = levene_results[key]["equal_var"]
            wt_vals   = animal_df.loc[animal_df["Genotype"].astype(str) == "WT",   value_col].dropna().values
            nlgf_vals = animal_df.loc[animal_df["Genotype"].astype(str) == "NLGF", value_col].dropna().values
            if len(wt_vals) < 2 or len(nlgf_vals) < 2:
                continue
            stat, p_val = ttest_ind(wt_vals, nlgf_vals, equal_var=bool(equal_var))
            test_name = "Student's t-test" if equal_var else "Welch's t-test"
        else:
            value_col = "metric_median"
            wt_vals   = animal_df.loc[animal_df["Genotype"].astype(str) == "WT",   value_col].dropna().values
            nlgf_vals = animal_df.loc[animal_df["Genotype"].astype(str) == "NLGF", value_col].dropna().values
            if len(wt_vals) < 1 or len(nlgf_vals) < 1:
                continue
            stat, p_val = mannwhitneyu(wt_vals, nlgf_vals, alternative="two-sided")
            test_name = "Mann-Whitney U"

        test_results.append({
            "metric": metric, "environment": env,
            "test": test_name, "value_col": value_col,
            "statistic": stat, "pvalue": p_val,
            "n_WT": len(wt_vals), "n_NLGF": len(nlgf_vals),
            "WT_value":   np.mean(wt_vals)   if is_normal else np.median(wt_vals),
            "NLGF_value": np.mean(nlgf_vals) if is_normal else np.median(nlgf_vals),
            "stars": _pstars(p_val),
        })

test_results_df = pd.DataFrame(test_results)
test_results_df.to_csv(os.path.join(OUTPUT_DIR, "primary_test_results.csv"), index=False)


def _colour_pval(p):
    if pd.isna(p): return ""
    if p < 0.001: return "background-color: #c62828; color: white; font-weight: bold;"
    if p < 0.01:  return "background-color: #ef5350; color: white; font-weight: bold;"
    if p < 0.05:  return "background-color: #ffcc80; color: black; font-weight: bold;"
    return ""


styled = (test_results_df.style
          .applymap(_colour_pval, subset=["pvalue"])
          .format({"statistic": "{:.4f}", "pvalue": "{:.4f}",
                   "WT_value": "{:.4f}", "NLGF_value": "{:.4f}"}))

display(HTML("<h3>Primary statistical test per metric × environment "
             "(t-test if normal, Mann-Whitney U otherwise)</h3>"))
display(styled)

html_path = os.path.join(OUTPUT_DIR, "primary_test_results.html")
with open(html_path, "w") as f:
    f.write(styled.to_html())

print(f"✅ Primary test results saved to: {os.path.join(OUTPUT_DIR, 'primary_test_results.csv')}")
print(f"✅ HTML visualisation saved to: {html_path}")

## Plot the same graphs, sharing the y-axis for the 2 environments, annotated with the Step 3 primary test

In [ ]:
# Shared y-axis plots for PAC comodulogram metrics, plotted at the same
# per-animal level Step 3 tested on, and annotated with whichever test Step 3
# selected for that metric × environment (t-test/Welch's -> mean ± SEM bars,
# Mann-Whitney U -> median/IQR bars).
# Run after the Step 1-3 cells above, where df, metrics, animal_agg_store and
# test_results_df are defined.

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SHARED_Y_OUTDIR = os.path.join(OUTPUT_DIR, "shared_yaxis_primary_test_plots")
os.makedirs(SHARED_Y_OUTDIR, exist_ok=True)

# Put open field first, linear track second if both exist
preferred_env_order = ["open_field", "linear_track"]
envs_present = list(df["environment"].dropna().astype(str).unique())
envs = [e for e in preferred_env_order if e in envs_present]
envs += [e for e in sorted(envs_present) if e not in envs]


def _clean_name(x):
    return str(x).replace("/", "_").replace(" ", "_")


def _ptext(test_name, p):
    if pd.isna(p):
        return f"{test_name} p=n/a"
    if p < 0.001:
        return f"{test_name} p<0.001"
    return f"{test_name} p={p:.3f}"


def plot_metric_shared_yaxis(metric, ylabel, save=True):
    envs_for_metric = [e for e in envs if (metric, e) in animal_agg_store]
    if not envs_for_metric:
        return

    # shared y-range across environments, computed on the value each test used
    all_vals = []
    for env in envs_for_metric:
        row = test_results_df[(test_results_df["metric"] == metric) &
                               (test_results_df["environment"] == env)]
        if row.empty:
            continue
        value_col = row.iloc[0]["value_col"]
        all_vals.append(animal_agg_store[(metric, env)][value_col].dropna().values)
    if not all_vals:
        return
    y_all = np.concatenate(all_vals)
    ymin, ymax = np.nanmin(y_all), np.nanmax(y_all)
    yrng = ymax - ymin if ymax > ymin else max(abs(ymax), 1.0) * 0.1

    y_bracket = ymax + 0.12 * yrng
    h_bracket = 0.04 * yrng
    y_upper = ymax + 0.30 * yrng
    y_lower = ymin - 0.08 * yrng

    fig, axes = plt.subplots(
        1, len(envs_for_metric),
        figsize=(1.65 * len(envs_for_metric), 2.2),
        sharey=True
    )
    if len(envs_for_metric) == 1:
        axes = [axes]

    rng = np.random.default_rng(10)

    for ax, env in zip(axes, envs_for_metric):
        row = test_results_df[(test_results_df["metric"] == metric) &
                               (test_results_df["environment"] == env)]
        if row.empty:
            ax.set_visible(False)
            continue
        row = row.iloc[0]
        value_col = row["value_col"]
        test_name = row["test"]

        animal_df = animal_agg_store[(metric, env)]
        genos = [g for g in GENO_ORDER if g in animal_df["Genotype"].astype(str).values]
        vals = {
            g: animal_df.loc[animal_df["Genotype"].astype(str) == g, value_col].dropna().values
            for g in genos
        }

        # Summary bars: mean ± SEM for t-test metrics, median/IQR for Mann-Whitney U
        for xi, g in enumerate(genos):
            if len(vals[g]) == 0:
                continue
            if value_col == "metric_mean":
                center = np.mean(vals[g])
                sem = np.std(vals[g], ddof=1) / np.sqrt(len(vals[g])) if len(vals[g]) > 1 else 0.0
                lo, hi = center - sem, center + sem
            else:
                lo = np.nanpercentile(vals[g], 25)
                hi = np.nanpercentile(vals[g], 75)
                center = np.nanmedian(vals[g])

            ax.hlines([lo, hi], xi - 0.16, xi + 0.16,
                      colors="black", linewidth=0.8, zorder=7)
            ax.hlines(center, xi - 0.24, xi + 0.24,
                      colors="black", linewidth=1.6, zorder=8)

        # Individual animal points
        for xi, g in enumerate(genos):
            y = vals[g]
            jitter = rng.normal(0, 0.045, size=len(y))
            ax.scatter(
                np.full(len(y), xi) + jitter, y,
                s=14, color=GENO_COLORS.get(g, "#aaaaaa"),
                edgecolor="black", linewidth=0.35, alpha=0.9, zorder=10,
            )

        # Primary-test annotation
        if len(genos) >= 2:
            p = row["pvalue"]
            stars = row["stars"]
            label = f"{stars}\n{_ptext(test_name, p)}"

            ax.plot([0, 0, 1, 1],
                    [y_bracket, y_bracket + h_bracket,
                     y_bracket + h_bracket, y_bracket],
                    color="black", linewidth=0.8, clip_on=False)
            ax.text(0.5, y_bracket + h_bracket, label,
                    ha="center", va="bottom", fontsize=6)

        ax.set_xticks(range(len(genos)))
        ax.set_xticklabels([f"{g}\n(n={len(vals[g])})" for g in genos])
        ax.set_title(env.replace("_", " "))
        ax.set_ylim(y_lower, y_upper)

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    axes[0].set_ylabel(ylabel)
    for ax in axes[1:]:
        ax.set_ylabel("")

    fig.suptitle(ylabel, fontsize=7)
    fig.tight_layout()

    if save:
        fig.savefig(
            os.path.join(
                SHARED_Y_OUTDIR,
                f"{_clean_name(metric)}_openfield_lineartrack_shared_yaxis_primary_test.pdf"
            )
        )

    plt.show()
    plt.close(fig)


for metric, ylabel in metrics.items():
    plot_metric_shared_yaxis(metric, ylabel)

print("Done.")
print(f"Saved shared y-axis plots to: {SHARED_Y_OUTDIR}")

## Generalized cell for all paired metrics with glmmTMB family selection:


- ✅ Loops over all paired metrics (PAC, n_oscillatory_epochs)
- ✅ Per environment (open_field, linear_track)
- ✅ Family selection (Gaussian vs Gamma vs Inverse Gaussian via AIC)
- ✅ Type III ANOVA + emmeans contrasts from best-fit family
- ✅ Organized output: Results saved to glmmTMB_paired_metrics/ subfolder
- ✅ Summary CSV: Shows which family won for each metric × environment combination

Results will include:

- {metric}_{env}_anova_{family}.csv
- {metric}_{env}_geno_by_band_{family}.csv
- {metric}_{env}_band_by_geno_{family}.csv
- {metric}_{env}_interaction_{family}.csv

In [5]:
# ── 2-way GLMM per environment for ALL PAIRED METRICS ────────────────────────
# Family selection: Gaussian vs Gamma vs Inverse Gaussian
# Metrics: PAC, n_oscillatory_epochs, theta_gamma_modulation_index (slow/fast variants)

import pandas as pd
import numpy as np
from pathlib import Path
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter

ro.r("suppressPackageStartupMessages(library(glmmTMB))")
ro.r("suppressPackageStartupMessages(library(car))")
ro.r("suppressPackageStartupMessages(library(emmeans))")

def sig_label(p):
    if pd.isna(p): return 'NA'
    if p < 0.001: return '***'
    if p < 0.01:  return '**'
    if p < 0.05:  return '*'
    return 'ns'

# ── Define all paired metrics (slow/fast pairs) ────────────────────────────────
PAIRED_METRICS = {
    'PAC': {
        'slow': 'comodulogram_PAC_slow_gamma',
        'fast': 'comodulogram_PAC_fast_gamma',
        'label': 'Theta–gamma PAC',
    },
    'n_oscillatory_epochs': {
        'slow': 'n_oscillatory_epochs_slow_gamm',
        'fast': 'n_oscillatory_epochs_fast_gamma',
        'label': 'N oscillatory epochs',
    },
    'theta_gamma_modulation_index': {
        'slow': 'theta_gamma_modulation_index_slow_gamma',
        'fast': 'theta_gamma_modulation_index_fast_gamma',
        'label': 'Theta–gamma modulation index',
    }
}

# Filter to metrics that exist in df
PAIRED_METRICS = {k: v for k, v in PAIRED_METRICS.items() 
                  if v['slow'] in df.columns and v['fast'] in df.columns}

if not PAIRED_METRICS:
    print("No paired metrics found in data")
else:
    print(f"Found {len(PAIRED_METRICS)} paired metrics: {list(PAIRED_METRICS.keys())}")

# ── Master results storage ─────────────────────────────────────────────────────
all_paired_results = {}
family_summary_all = []

# ── Loop over each paired metric ────────────────────────────────────────────────
for metric_name, metric_info in PAIRED_METRICS.items():
    print(f"\n\n{'='*80}")
    print(f"METRIC: {metric_name} ({metric_info['label']})")
    print(f"{'='*80}")
    
    all_paired_results[metric_name] = {}
    
    # ── Loop over each environment ─────────────────────────────────────────
    for env in sorted(df['environment'].dropna().unique()):
        print(f"\n{'─'*80}\nENVIRONMENT: {env}\n{'─'*80}")
        
        # Subset to this environment
        sub_df = df[df['environment'] == env].copy()
        
        # Build long-format (slow + fast) for this metric
        slow = sub_df[['mouse_name', 'Genotype', metric_info['slow']]].rename(
            columns={metric_info['slow']: 'value'})
        slow['gamma_band'] = 'slow'
        
        fast = sub_df[['mouse_name', 'Genotype', metric_info['fast']]].rename(
            columns={metric_info['fast']: 'value'})
        fast['gamma_band'] = 'fast'
        
        long_data = pd.concat([slow, fast], ignore_index=True).dropna(subset=['value'])
        
        # Handle zeros/negatives: add small offset for Gamma/IG families
        min_positive = long_data.loc[long_data['value'] > 0, 'value'].min()
        offset = min_positive / 2 if pd.notna(min_positive) else 1e-5
        long_data['value_offset'] = long_data['value'].apply(lambda x: x if x > 0 else offset)
        
        long_data['mouse_name']  = long_data['mouse_name'].astype(str)
        long_data['Genotype']    = long_data['Genotype'].astype(str)
        long_data['gamma_band']  = long_data['gamma_band'].astype(str)
        
        print(f"n rows = {len(long_data)}, n mice = {long_data['mouse_name'].nunique()}")
        print(f"{metric_name} range: [{long_data['value'].min():.6f}, {long_data['value'].max():.6f}]")
        print(f"Offset for zeros: {offset:.2e}")
        print(long_data.groupby(['Genotype', 'gamma_band'])['value'].agg(['count', 'median']))
        
        if long_data['mouse_name'].nunique() < 4:
            print("  ⚠️  Skipped: insufficient data")
            continue
        
        # ── Pass to R and fit GLMM with family selection ─────────────────────
        with localconverter(ro.default_converter + pandas2ri.converter):
            r_df = ro.conversion.py2rpy(long_data)
        ro.globalenv['dat'] = r_df
        
        r_code = """
        suppressWarnings({
            dat$mouse_name  <- factor(dat$mouse_name)
            dat$Genotype    <- factor(dat$Genotype,    levels = c("WT", "NLGF"))
            dat$gamma_band  <- factor(dat$gamma_band,  levels = c("slow", "fast"))
            
            contrasts(dat$Genotype)   <- contr.sum(nlevels(dat$Genotype))
            contrasts(dat$gamma_band) <- contr.sum(nlevels(dat$gamma_band))
            
            # ── Fit three families ──────────────────────────────────────────────
            fit_gaussian <- glmmTMB(
                value ~ Genotype * gamma_band + (1 | mouse_name),
                data = dat,
                family = gaussian()
            )
            
            fit_gamma <- glmmTMB(
                value_offset ~ Genotype * gamma_band + (1 | mouse_name),
                data = dat,
                family = Gamma(link = "log")
            )
            
            fit_igaussian <- glmmTMB(
                value_offset ~ Genotype * gamma_band + (1 | mouse_name),
                data = dat,
                family = inverse.gaussian(link = "log")
            )
            
            # ── Compare AICs ───────────────────────────────────────────────────
            aic_gaussian   <- AIC(fit_gaussian)
            aic_gamma      <- AIC(fit_gamma)
            aic_igaussian  <- AIC(fit_igaussian)
            
            aic_list <- c(gaussian = aic_gaussian, gamma = aic_gamma, igaussian = aic_igaussian)
            best_family_name <- names(which.min(aic_list))
            best_fit <- if (best_family_name == "gaussian") fit_gaussian
                       else if (best_family_name == "gamma") fit_gamma
                       else fit_igaussian
            
            delta_aic_min <- min(aic_list)
            delta_aic_from_best <- aic_list - delta_aic_min
            
            # ── Extract results from best fit via car::Anova (Type III) ─────────
            aov_table <- as.data.frame(Anova(best_fit, type = 3))
            aov_table <- data.frame(term = rownames(aov_table), aov_table,
                                    row.names = NULL, check.names = FALSE)
            
            # 1. Genotype contrast within each gamma band
            emm_geno_by_band <- emmeans(best_fit, ~ Genotype | gamma_band)
            geno_by_band_df  <- as.data.frame(contrast(emm_geno_by_band, method = "pairwise"))
            
            # 2. Gamma band contrast within each genotype
            emm_band_by_geno <- emmeans(best_fit, ~ gamma_band | Genotype)
            band_by_geno_df  <- as.data.frame(contrast(emm_band_by_geno, method = "pairwise"))
            
            # 3. Interaction contrast
            emm_full <- emmeans(best_fit, ~ Genotype * gamma_band)
            interaction_df <- as.data.frame(contrast(emm_full, interaction = "pairwise"))
            
            # Extract random effect variance and model fit info
            re_var  <- as.numeric(VarCorr(best_fit)$cond$mouse_name[1,1])
            aic_best <- AIC(best_fit)
            
            list(
                best_family        = best_family_name,
                aic_gaussian       = aic_gaussian,
                aic_gamma          = aic_gamma,
                aic_igaussian      = aic_igaussian,
                delta_aics         = delta_aic_from_best,
                aic_best           = aic_best,
                anova_table        = aov_table,
                geno_by_band       = geno_by_band_df,
                band_by_geno       = band_by_geno_df,
                interaction        = interaction_df,
                re_var             = re_var
            )
        })
        """
        
        try:
            out = ro.r(r_code)
            
            with localconverter(ro.default_converter + pandas2ri.converter):
                aov_df          = ro.conversion.rpy2py(out.rx2('anova_table'))
                geno_by_band    = ro.conversion.rpy2py(out.rx2('geno_by_band'))
                band_by_geno    = ro.conversion.rpy2py(out.rx2('band_by_geno'))
                interaction_df  = ro.conversion.rpy2py(out.rx2('interaction'))
            
            best_family = str(out.rx2('best_family')[0])
            aic_gaussian = float(out.rx2('aic_gaussian')[0])
            aic_gamma = float(out.rx2('aic_gamma')[0])
            aic_igaussian = float(out.rx2('aic_igaussian')[0])
            aic_best = float(out.rx2('aic_best')[0])
            re_var = float(out.rx2('re_var')[0])
            
            print(f"\n{'─'*70}")
            print(f"FAMILY SELECTION (Type III via car::Anova):")
            print(f"  Gaussian:        AIC = {aic_gaussian:.2f}")
            print(f"  Gamma:           AIC = {aic_gamma:.2f}")
            print(f"  Inverse Gaussian: AIC = {aic_igaussian:.2f}")
            print(f"  ✓ Best fit:      {best_family.upper()} (AIC = {aic_best:.2f})")
            print(f"{'─'*70}")
            
            print(f"\nModel fit (best family = {best_family}):")
            print(f"  AIC: {aic_best:.2f}  |  RE variance (mouse): {re_var:.6f}")
            
            print(f"\n─── Type III ANOVA ({best_family}) ───────────────────────────────────")
            print(aov_df.to_string(index=False))
            
            print(f"\n─── Genotype (NLGF vs WT) within each gamma band ────────")
            print(geno_by_band.to_string(index=False))
            
            print(f"\n─── Gamma band (fast vs slow) within each genotype ──────")
            print(band_by_geno.to_string(index=False))
            
            print(f"\n─── Interaction contrast ──────────────────────────────────")
            print(interaction_df.to_string(index=False))
            
            # Store for later reference
            all_paired_results[metric_name][env] = {
                'family': best_family,
                'aic_gaussian': aic_gaussian,
                'aic_gamma': aic_gamma,
                'aic_igaussian': aic_igaussian,
                'anova': aov_df,
                'geno_by_band': geno_by_band,
                'band_by_geno': band_by_geno,
                'interaction': interaction_df,
            }
            
            family_summary_all.append({
                'metric': metric_name,
                'environment': env,
                'best_family': best_family,
                'aic_gaussian': aic_gaussian,
                'aic_gamma': aic_gamma,
                'aic_igaussian': aic_igaussian,
                'n_mice': long_data['mouse_name'].nunique(),
            })
            
            # Save results
            env_clean = env.replace(" ", "_")
            metric_clean = metric_name.replace(" ", "_")
            out_dir = Path(OUTPUT_DIR) / 'glmmTMB_paired_metrics'
            out_dir.mkdir(parents=True, exist_ok=True)
            
            aov_df.to_csv(out_dir / f"{metric_clean}_{env_clean}_anova_{best_family}.csv", index=False)
            geno_by_band.to_csv(out_dir / f"{metric_clean}_{env_clean}_geno_by_band_{best_family}.csv", index=False)
            band_by_geno.to_csv(out_dir / f"{metric_clean}_{env_clean}_band_by_geno_{best_family}.csv", index=False)
            interaction_df.to_csv(out_dir / f"{metric_clean}_{env_clean}_interaction_{best_family}.csv", index=False)
            
            print(f"\n✅ Saved results for {metric_name} × {env} (family: {best_family})")
            
        except Exception as e:
            print(f"❌ FAILED: {e}")
            import traceback
            traceback.print_exc()

# ── Summary: families selected per metric × environment ────────────────────────
print(f"\n\n{'='*80}")
print("SUMMARY: FAMILIES SELECTED FOR ALL PAIRED METRICS")
print(f"{'='*80}")

if family_summary_all:
    summary_df = pd.DataFrame(family_summary_all)
    print(summary_df.to_string(index=False))
    
    # Save summary
    summary_path = Path(OUTPUT_DIR) / 'glmmTMB_paired_metrics' / "family_selection_summary_all_paired.csv"
    summary_df.to_csv(summary_path, index=False)
    print(f"\n✅ Family selection summary saved to: {summary_path}")

print(f"\n✅ All paired metrics analyzed and saved to: {Path(OUTPUT_DIR) / 'glmmTMB_paired_metrics'}")

R callback write-console: In addition:   
R callback write-console: Warning message:
  
R callback write-console: In check_dep_version(dep_pkg = "TMB") :  
R callback write-console:  package version mismatch: 
glmmTMB was built with TMB package version 1.9.19
Current TMB package version is 1.9.21
Please re-install glmmTMB from source or restore original ‘TMB’ package (see '?reinstalling' for more information)
  


Found 3 paired metrics: ['PAC', 'n_oscillatory_epochs', 'theta_gamma_modulation_index']


METRIC: PAC (Theta–gamma PAC)

────────────────────────────────────────────────────────────────────────────────
ENVIRONMENT: linear_track
────────────────────────────────────────────────────────────────────────────────
n rows = 34, n mice = 17
PAC range: [0.000043, 0.003230]
Offset for zeros: 2.16e-05
                     count    median
Genotype gamma_band                 
NLGF     fast            9  0.000445
         slow            9  0.000188
WT       fast            8  0.001543
         slow            8  0.000687

──────────────────────────────────────────────────────────────────────
FAMILY SELECTION (Type III via car::Anova):
  Gaussian:        AIC = -383.68
  Gamma:           AIC = -425.98
  Inverse Gaussian: AIC = -403.90
  ✓ Best fit:      GAMMA (AIC = -425.98)
──────────────────────────────────────────────────────────────────────

Model fit (best family = gamma):
  AIC: -425.98  |  RE v

## Compare GLMM-TMB vs Mann-Whitney U: concordance of genotype effects


This comparison uses the environment-specific GLMM-TMB genotype contrasts and compares them with Mann-Whitney U genotype tests for the same environment and original metric column.

For paired slow/fast metrics, GLMM-TMB rows are mapped back to the matching original slow/fast columns before merging with Mann-Whitney results.


In [ ]:
# -- Compare GLMM-TMB vs Step 3 secondary test: genotype contrast concordance -----
# Step 3 (cell 6/7) picks a per-metric x environment secondary test (t-test/Welch's
# when normal, Mann-Whitney U otherwise), so this is no longer always MWU -- see
# the 'secondary_test' column below for which test backs each row.
from pathlib import Path
from IPython.display import display, HTML
import os
import numpy as np
import pandas as pd

ALPHA = 0.05
GLMM_DIR = Path(OUTPUT_DIR) / "glmmTMB_paired_metrics"
SECONDARY_PATH = Path(OUTPUT_DIR) / "primary_test_results.csv"

PAIRED_ORIGINAL_METRIC = {
    ("PAC", "slow"): "comodulogram_PAC_slow_gamma",
    ("PAC", "fast"): "comodulogram_PAC_fast_gamma",
    ("n_oscillatory_epochs", "slow"): "n_oscillatory_epochs_slow_gamm",
    ("n_oscillatory_epochs", "fast"): "n_oscillatory_epochs_fast_gamma",
    ("theta_gamma_modulation_index", "slow"): "theta_gamma_modulation_index_slow_gamma",
    ("theta_gamma_modulation_index", "fast"): "theta_gamma_modulation_index_fast_gamma",
}

PRETTY_METRIC = {
    "comodulogram_PAC_slow_gamma": "PAC slow",
    "comodulogram_PAC_fast_gamma": "PAC fast",
    "n_oscillatory_epochs_slow_gamm": "N oscillatory epochs slow",
    "n_oscillatory_epochs_fast_gamma": "N oscillatory epochs fast",
    "theta_gamma_modulation_index_slow_gamma": "Theta–Slow gamma modulation index",
    "theta_gamma_modulation_index_fast_gamma": "Theta–Fast gamma modulation index",
}

def sig_label(p):
    if pd.isna(p): return "NA"
    if p < 0.001: return "***"
    if p < 0.01:  return "**"
    if p < 0.05:  return "*"
    return "ns"

def clean_key(value):
    return str(value).strip().lower().replace(" ", "_").replace("/", "_")

def p_value_column(table):
    for candidate in ["p.value", "pvalue", "Pr(>|z|)", "Pr(>|t|)", "p"]:
        if candidate in table.columns:
            return candidate
    candidates = [col for col in table.columns if "p" in str(col).lower()]
    return candidates[0] if candidates else None

def stat_column(table):
    for candidate in ["z.ratio", "t.ratio", "statistic"]:
        if candidate in table.columns:
            return candidate
    return None

def parse_geno_filename(path, env_lookup):
    """Parse {metric}_{environment}_geno_by_band_{family}.csv."""
    stem = path.name.removesuffix(".csv")
    if "_geno_by_band_" not in stem:
        return None

    for env_clean, env_label in sorted(env_lookup.items(), key=lambda item: len(item[0]), reverse=True):
        marker = f"_{env_clean}_geno_by_band_"
        if marker in stem:
            metric_name, family = stem.split(marker, 1)
            return metric_name, env_label, family
    return None

def extract_glmm_rows_from_table(table, metric_name, environment, family):
    rows = []
    p_col = p_value_column(table)
    s_col = stat_column(table)
    if p_col is None:
        return rows

    for _, row in table.iterrows():
        if "gamma_band" in table.columns and pd.notna(row.get("gamma_band")):
            gamma_band = str(row.get("gamma_band")).strip().lower()
        else:
            contrast_text = str(row.get("contrast", "")).lower()
            if "slow" in contrast_text:
                gamma_band = "slow"
            elif "fast" in contrast_text:
                gamma_band = "fast"
            else:
                continue

        original_metric = PAIRED_ORIGINAL_METRIC.get((metric_name, gamma_band))
        if original_metric is None:
            continue

        rows.append({
            "metric_group": metric_name,
            "gamma_band": gamma_band,
            "original_metric": original_metric,
            "metric": PRETTY_METRIC.get(original_metric, original_metric),
            "environment": environment,
            "environment_key": clean_key(environment),
            "glmm_family": family,
            "glmm_contrast": row.get("contrast", np.nan),
            "glmm_estimate": row.get("estimate", np.nan),
            "glmm_SE": row.get("SE", np.nan),
            "glmm_df": row.get("df", np.nan),
            "glmm_stat": row.get(s_col, np.nan) if s_col is not None else np.nan,
            "glmm_p": pd.to_numeric(row.get(p_col), errors="coerce"),
        })
    return rows

def extract_glmm_rows():
    rows = []

    if "all_paired_results" in globals() and isinstance(all_paired_results, dict) and all_paired_results:
        for metric_name, env_dict in all_paired_results.items():
            for environment, env_data in env_dict.items():
                table = env_data.get("geno_by_band")
                if table is None or len(table) == 0:
                    continue
                family = env_data.get("family", np.nan)
                rows.extend(extract_glmm_rows_from_table(table, metric_name, environment, family))

    if rows:
        return pd.DataFrame(rows)

    if not GLMM_DIR.exists():
        raise FileNotFoundError(f"GLMM-TMB results folder not found: {GLMM_DIR}")

    env_lookup = {clean_key(env): env for env in sorted(df["environment"].dropna().unique())}
    for path in sorted(GLMM_DIR.glob("*_geno_by_band_*.csv")):
        parsed = parse_geno_filename(path, env_lookup)
        if parsed is None:
            print(f"Skipping unrecognized GLMM filename: {path.name}")
            continue
        metric_name, environment, family = parsed
        table = pd.read_csv(path)
        rows.extend(extract_glmm_rows_from_table(table, metric_name, environment, family))

    return pd.DataFrame(rows)

# Load Step 3 secondary-test results from memory when available, otherwise from disk.
if "test_results_df" in globals() and isinstance(test_results_df, pd.DataFrame):
    secondary_df = test_results_df.copy()
elif SECONDARY_PATH.exists():
    secondary_df = pd.read_csv(SECONDARY_PATH)
else:
    raise FileNotFoundError(f"Step 3 secondary-test results not found: {SECONDARY_PATH}")

glmm_df = extract_glmm_rows()
if glmm_df.empty:
    raise ValueError("No GLMM-TMB genotype-by-gamma-band contrast rows were extracted.")

required_secondary_cols = [
    "metric", "environment", "test", "n_WT", "n_NLGF", "WT_value", "NLGF_value", "statistic", "pvalue"
]
missing = [col for col in required_secondary_cols if col not in secondary_df.columns]
if missing:
    raise KeyError(f"Step 3 secondary-test table is missing required columns: {missing}")

secondary_for_comparison = (
    secondary_df.loc[secondary_df["metric"].isin(PAIRED_ORIGINAL_METRIC.values()), required_secondary_cols]
    .rename(columns={"metric": "original_metric", "test": "secondary_test",
                      "statistic": "secondary_statistic", "pvalue": "secondary_p"})
    .copy()
)
secondary_for_comparison["environment_key"] = secondary_for_comparison["environment"].map(clean_key)

comp = pd.merge(
    glmm_df,
    secondary_for_comparison,
    on=["original_metric", "environment_key"],
    how="inner",
    suffixes=("", "_secondary"),
)

if comp.empty:
    raise ValueError(
        "No GLMM-TMB rows matched Mann-Whitney rows. Check environment labels and metric names."
    )

# Keep GLMM environment label and drop duplicate secondary-test environment label.
if "environment_secondary" in comp.columns:
    comp = comp.drop(columns=["environment_secondary"])

comp["glmm_sig"] = comp["glmm_p"] < ALPHA
comp["secondary_sig"] = comp["secondary_p"] < ALPHA
comp["glmm_sig_label"] = comp["glmm_p"].map(sig_label)
comp["secondary_sig_label"] = comp["secondary_p"].map(sig_label)

def classify(row):
    if row["glmm_sig"] and row["secondary_sig"]:
        return "both_significant"
    if row["glmm_sig"] and not row["secondary_sig"]:
        return "glmmTMB_only"
    if (not row["glmm_sig"]) and row["secondary_sig"]:
        return "secondary_only"
    return "both_not_significant"

comp["concordance"] = comp.apply(classify, axis=1)
comp = comp.sort_values(["environment", "metric_group", "gamma_band"]).reset_index(drop=True)

full_path = Path(OUTPUT_DIR) / "glmmTMB_vs_secondary_test_concordance_by_gamma_band.csv"
sig_path = Path(OUTPUT_DIR) / "glmmTMB_vs_secondary_test_significant_results_by_gamma_band.csv"
comp.to_csv(full_path, index=False)
comp.loc[comp["glmm_sig"] | comp["secondary_sig"]].to_csv(sig_path, index=False)

print(f"Extracted GLMM-TMB rows: {len(glmm_df)}")
print(f"Matched comparisons: {len(comp)}")
print(f"Saved full concordance table: {full_path}")
print(f"Saved significant-results table: {sig_path}")

print("\nConcordance counts:")
print(comp["concordance"].value_counts(dropna=False).to_string())

print("\nConcordance by environment:")
print(pd.crosstab(comp["environment"], comp["concordance"]).to_string())

print("\nConcordance by gamma band:")
print(pd.crosstab(comp["gamma_band"], comp["concordance"]).to_string())

show_cols = [
    "metric", "environment", "gamma_band", "glmm_family",
    "glmm_estimate", "glmm_p", "glmm_sig_label",
    "secondary_test", "secondary_p", "secondary_sig_label", "secondary_statistic",
    "WT_value", "NLGF_value", "n_WT", "n_NLGF", "concordance",
]

display(HTML("<h3>GLMM-TMB vs Step 3 secondary test: genotype-effect concordance</h3>"))
display(
    comp[show_cols].style
    .format({
        "glmm_estimate": "{:.6f}",
        "glmm_p": "{:.4f}",
        "secondary_p": "{:.4f}",
        "secondary_statistic": "{:.2f}",
        "WT_value": "{:.6f}",
        "NLGF_value": "{:.6f}",
    })
    .set_properties(**{"font-size": "11px"})
)

glmmTMB_vs_secondary_test = comp


## FDR Correction Across MLM vs Secondary-Test Results

The concordance table above compares raw p-values across metric × environment combinations (12 tests here). Running that many tests without correction inflates the false-positive rate — a handful of p-values in the 0.04-0.06 range are expected by chance alone.

This cell applies **Benjamini-Hochberg FDR correction** separately to the MLM p-values and the Step 3 secondary-test p-values (t-test/Welch's or Mann-Whitney U, per metric × environment — each family corrected on its own, since they are different test families answering different statistical questions), then re-classifies concordance using the corrected significance calls.

**Interpretation:**
- `both_fdr` — significant in both tests after correction → the most trustworthy findings
- `GLMM-TMB_only_fdr` / `secondary_only_fdr` — significant in one test but not the other after correction → treat as borderline; check for outlier-driven effects (e.g. via the animal-level violin plots) or run a robustness check (e.g. median-based permutation test)
- `neither_fdr` — not significant in either after correction

Re-run this cell any time `comp` is rebuilt (e.g. after adding new metrics).

In [ ]:
# ── FDR correction (Benjamini-Hochberg) across all GLMM-TMB vs secondary-test results ───────────
# Corrects for running one test per metric × environment combination.
# Adds *_p_fdr and *_sig_fdr columns to `comp`, plus a corrected concordance
# label. Depends on `comp` from the concordance cell above — re-run this any
# time `comp` is rebuilt (e.g. after adding new metrics).

from statsmodels.stats.multitest import multipletests

ALPHA = 0.05

def apply_fdr_correction(comp_df, alpha=ALPHA):
    """
    Applies Benjamini-Hochberg FDR correction separately to the GLMM-TMB and Step 3
    secondary-test (t-test/Welch's or Mann-Whitney U, per metric x environment)
    p-value columns of the concordance table, and re-classifies concordance
    using the corrected p-values.

    The two families (GLMM-TMB, secondary test) are corrected independently, since
    they are different test types answering different statistical questions —
    pooling them into one correction would not be appropriate.
    """
    comp_df = comp_df.copy()

    # GLMM-TMB correction
    glmm_mask = comp_df['glmm_p'].notna()
    comp_df.loc[glmm_mask, 'glmm_p_fdr'] = multipletests(
        comp_df.loc[glmm_mask, 'glmm_p'], method='fdr_bh'
    )[1]
    comp_df['glmm_sig_fdr'] = comp_df['glmm_p_fdr'] < alpha

    # Secondary-test correction
    secondary_mask = comp_df['secondary_p'].notna()
    comp_df.loc[secondary_mask, 'secondary_p_fdr'] = multipletests(
        comp_df.loc[secondary_mask, 'secondary_p'], method='fdr_bh'
    )[1]
    comp_df['secondary_sig_fdr'] = comp_df['secondary_p_fdr'] < alpha

    # Re-classify concordance on FDR-corrected p-values
    def classify_fdr(row):
        g = bool(row['glmm_sig_fdr']) if pd.notna(row.get('glmm_sig_fdr')) else False
        s = bool(row['secondary_sig_fdr']) if pd.notna(row.get('secondary_sig_fdr')) else False
        if g and s:
            return 'both_fdr'
        if g:
            return 'GLMM-TMB_only_fdr'
        if s:
            return 'secondary_only_fdr'
        return 'neither_fdr'

    comp_df['concordance_fdr'] = comp_df.apply(classify_fdr, axis=1)
    return comp_df


comp_fdr = apply_fdr_correction(comp)

display(HTML("<h3>GLMM-TMB vs Step 3 secondary test — after Benjamini-Hochberg FDR correction</h3>"))
show_cols = ['metric', 'environment',
             'glmm_p', 'glmm_p_fdr', 'glmm_sig_fdr',
             'secondary_p', 'secondary_p_fdr', 'secondary_sig_fdr',
             'concordance', 'concordance_fdr']

def _colour_sig(row):
    g_sig = bool(row.get('glmm_sig_fdr'))
    s_sig = bool(row.get('secondary_sig_fdr'))
    if g_sig and s_sig:
        bg = "#c8e6c9"   # green — robust
    elif g_sig or s_sig:
        bg = "#fff9c4"   # yellow — borderline / method-dependent
    else:
        bg = ""
    return [f"background-color:{bg}" if bg else "" for _ in row]

display(
    comp_fdr[show_cols]
        .sort_values(['environment', 'metric'])
        .style
        .apply(_colour_sig, axis=1)
        .format({
            'glmm_p':     '{:.4f}',
            'glmm_p_fdr': '{:.4f}',
            'secondary_p':     '{:.4f}',
            'secondary_p_fdr': '{:.4f}',
        })
        .set_properties(**{'font-size': '11px'})
)

print("\n── Concordance counts BEFORE FDR correction ────────────────────────────")
print(comp_fdr['concordance'].value_counts().to_string())
print("\n── Concordance counts AFTER FDR correction ─────────────────────────────")
print(comp_fdr['concordance_fdr'].value_counts().to_string())

# ── Save ──────────────────────────────────────────────────────────────────────
fdr_path = os.path.join(OUTPUT_DIR, "glmm-tmb_vs_secondary_test_concordance_FDR.csv")
comp_fdr.to_csv(fdr_path, index=False)
print(f"\n✅ Saved: {fdr_path}")

## Median-based permutation test Apply your median-based permutation test 
-specific discordant cells as a tiebreaker 
— it'll tell you whether the GLMM or the Step 3 secondary test is closer to the "outlier-robust truth."

In [ ]:
from pathlib import Path
from itertools import combinations
import numpy as np
import pandas as pd

ALPHA = 0.05

base = Path(
    "/Users/loukia/Library/CloudStorage/Dropbox-UCL/Loukia Katsouri/"
    "DataProtocolsEquipment/Ephys_Analysis/RobinData/Analysis/LFP_analysis"
)

data_path = base / "concatenated_lfp_stats.csv"
concordance_path = base / "glmmTMB_vs_secondary_test_concordance_by_gamma_band.csv"
out_path = base / "median_permutation_tiebreaker_discordant_glmmTMB_vs_secondary_test.csv"

lfp = pd.read_csv(data_path)
comp = pd.read_csv(concordance_path)

discordant = comp[comp["concordance"].isin(["glmmTMB_only", "secondary_only"])].copy()


def exact_median_permutation(values, labels, wt_label="WT", ko_label="NLGF"):
    values = np.asarray(values, dtype=float)
    labels = np.asarray(labels).astype(str)

    keep = np.isfinite(values) & np.isin(labels, [wt_label, ko_label])
    values = values[keep]
    labels = labels[keep]

    n_total = len(values)
    n_wt = int(np.sum(labels == wt_label))
    n_ko = int(np.sum(labels == ko_label))

    if n_wt == 0 or n_ko == 0:
        return None

    obs = float(
        np.nanmedian(values[labels == wt_label])
        - np.nanmedian(values[labels == ko_label])
    )
    abs_obs = abs(obs)

    diffs = []
    for wt_idx in combinations(range(n_total), n_wt):
        wt_mask = np.zeros(n_total, dtype=bool)
        wt_mask[list(wt_idx)] = True

        diff = float(
            np.nanmedian(values[wt_mask])
            - np.nanmedian(values[~wt_mask])
        )
        diffs.append(diff)

    diffs = np.asarray(diffs, dtype=float)

    return {
        "median_perm_stat_WT_minus_NLGF": obs,
        "median_perm_p_two_sided": float(np.mean(np.abs(diffs) >= abs_obs - 1e-15)),
        "median_perm_n_permutations": int(len(diffs)),
        "median_perm_null_mean": float(np.mean(diffs)),
        "median_perm_null_sd": float(np.std(diffs, ddof=1)),
        "median_perm_null_abs_ge_obs": int(np.sum(np.abs(diffs) >= abs_obs - 1e-15)),
        "median_perm_ci2p5": float(np.quantile(diffs, 0.025)),
        "median_perm_ci97p5": float(np.quantile(diffs, 0.975)),
    }


rows = []

for _, row in discordant.iterrows():
    env = row["environment"]
    metric_col = row["original_metric"]

    sub = lfp.loc[
        lfp["environment"].astype(str).eq(str(env)),
        ["mouse_name", "Genotype", metric_col],
    ].dropna()

    res = exact_median_permutation(
        sub[metric_col].to_numpy(),
        sub["Genotype"].to_numpy(),
    )

    if res is None:
        continue

    median_sig = res["median_perm_p_two_sided"] < ALPHA

    if median_sig == bool(row["glmm_sig"]) and median_sig != bool(row["secondary_sig"]):
        closer = "GLMM-TMB"
    elif median_sig == bool(row["secondary_sig"]) and median_sig != bool(row["glmm_sig"]):
        closer = "secondary_test"
    elif median_sig == bool(row["glmm_sig"]) == bool(row["secondary_sig"]):
        closer = "both"
    else:
        closer = "neither"

    rows.append({
        "metric_group": row["metric_group"],
        "gamma_band": row["gamma_band"],
        "original_metric": metric_col,
        "metric": row["metric"],
        "environment": env,
        "discordance": row["concordance"],
        "glmm_family": row["glmm_family"],
        "glmm_contrast": row["glmm_contrast"],
        "glmm_estimate": row["glmm_estimate"],
        "glmm_p": row["glmm_p"],
        "glmm_sig": bool(row["glmm_sig"]),
        "secondary_test": row["secondary_test"],
        "secondary_p": row["secondary_p"],
        "secondary_sig": bool(row["secondary_sig"]),
        "n_WT": int(row["n_WT"]),
        "n_NLGF": int(row["n_NLGF"]),
        "WT_value": row["WT_value"],
        "NLGF_value": row["NLGF_value"],
        **res,
        "median_perm_sig": median_sig,
        "tiebreaker_closer_to_median_truth": closer,
    })


out = (
    pd.DataFrame(rows)
    .sort_values(["environment", "metric_group", "gamma_band"])
    .reset_index(drop=True)
)

out.to_csv(out_path, index=False)

print(f"Saved: {out_path}")
print(
    out[
        [
            "environment",
            "metric",
            "gamma_band",
            "discordance",
            "glmm_p",
            "secondary_p",
            "median_perm_p_two_sided",
            "median_perm_sig",
            "tiebreaker_closer_to_median_truth",
        ]
    ].to_string(index=False)
)

In [ ]:
from pathlib import Path
from itertools import combinations
import numpy as np
import pandas as pd

ALPHA = 0.05

base = Path(
    "/Users/loukia/Library/CloudStorage/Dropbox-UCL/Loukia Katsouri/"
    "DataProtocolsEquipment/Ephys_Analysis/RobinData/Analysis/LFP_analysis"
)

data_path = base / "concatenated_lfp_stats.csv"
concordance_path = base / "glmm-tmb_vs_secondary_test_concordance_FDR.csv"
out_path = base / "median_permutation_tiebreaker_discordant_glmmTMB_vs_secondary_test_FDR.csv"

lfp = pd.read_csv(data_path)
comp = pd.read_csv(concordance_path)

discordant = comp[comp["concordance_fdr"].isin(["GLMM-TMB_only_fdr", "secondary_only_fdr"])].copy()


def exact_median_permutation(values, labels, wt_label="WT", ko_label="NLGF"):
    values = np.asarray(values, dtype=float)
    labels = np.asarray(labels).astype(str)

    keep = np.isfinite(values) & np.isin(labels, [wt_label, ko_label])
    values = values[keep]
    labels = labels[keep]

    n_total = len(values)
    n_wt = int(np.sum(labels == wt_label))
    n_ko = int(np.sum(labels == ko_label))

    if n_wt == 0 or n_ko == 0:
        return None

    obs = float(
        np.nanmedian(values[labels == wt_label])
        - np.nanmedian(values[labels == ko_label])
    )
    abs_obs = abs(obs)

    diffs = []
    for wt_idx in combinations(range(n_total), n_wt):
        wt_mask = np.zeros(n_total, dtype=bool)
        wt_mask[list(wt_idx)] = True

        diff = float(
            np.nanmedian(values[wt_mask])
            - np.nanmedian(values[~wt_mask])
        )
        diffs.append(diff)

    diffs = np.asarray(diffs, dtype=float)

    return {
        "median_perm_stat_WT_minus_NLGF": obs,
        "median_perm_p_two_sided": float(np.mean(np.abs(diffs) >= abs_obs - 1e-15)),
        "median_perm_n_permutations": int(len(diffs)),
        "median_perm_null_mean": float(np.mean(diffs)),
        "median_perm_null_sd": float(np.std(diffs, ddof=1)),
        "median_perm_null_abs_ge_obs": int(np.sum(np.abs(diffs) >= abs_obs - 1e-15)),
        "median_perm_ci2p5": float(np.quantile(diffs, 0.025)),
        "median_perm_ci97p5": float(np.quantile(diffs, 0.975)),
    }


rows = []

for _, row in discordant.iterrows():
    env = row["environment"]
    metric_col = row["original_metric"]

    sub = lfp.loc[
        lfp["environment"].astype(str).eq(str(env)),
        ["mouse_name", "Genotype", metric_col],
    ].dropna()

    res = exact_median_permutation(
        sub[metric_col].to_numpy(),
        sub["Genotype"].to_numpy(),
    )

    if res is None:
        continue

    median_sig = res["median_perm_p_two_sided"] < ALPHA

    if median_sig == bool(row["glmm_sig"]) and median_sig != bool(row["secondary_sig"]):
        closer = "GLMM-TMB"
    elif median_sig == bool(row["secondary_sig"]) and median_sig != bool(row["glmm_sig"]):
        closer = "secondary_test"
    elif median_sig == bool(row["glmm_sig"]) == bool(row["secondary_sig"]):
        closer = "both"
    else:
        closer = "neither"

    rows.append({
        "metric_group": row["metric_group"],
        "gamma_band": row["gamma_band"],
        "original_metric": metric_col,
        "metric": row["metric"],
        "environment": env,
        "discordance": row["concordance_fdr"],
        "glmm_family": row["glmm_family"],
        "glmm_contrast": row["glmm_contrast"],
        "glmm_estimate": row["glmm_estimate"],
        "glmm_p": row["glmm_p"],
        "glmm_sig": bool(row["glmm_sig"]),
        "secondary_test": row["secondary_test"],
        "secondary_p": row["secondary_p"],
        "secondary_sig": bool(row["secondary_sig"]),
        "n_WT": int(row["n_WT"]),
        "n_NLGF": int(row["n_NLGF"]),
        "WT_value": row["WT_value"],
        "NLGF_value": row["NLGF_value"],
        **res,
        "median_perm_sig": median_sig,
        "tiebreaker_closer_to_median_truth": closer,
    })


out = (
    pd.DataFrame(rows)
    .sort_values(["environment", "metric_group", "gamma_band"])
    .reset_index(drop=True)
)

out.to_csv(out_path, index=False)

print(f"Saved: {out_path}")
print(
    out[
        [
            "environment",
            "metric",
            "gamma_band",
            "discordance",
            "glmm_p",
            "secondary_p",
            "median_perm_p_two_sided",
            "median_perm_sig",
            "tiebreaker_closer_to_median_truth",
        ]
    ].to_string(index=False)
)

In [ ]:
# -- Significant GLMM-TMB / Step 3 secondary-test results summary -------------------
from pathlib import Path
from IPython.display import display, HTML
import pandas as pd

ALPHA = 0.05
summary_path = Path(OUTPUT_DIR) / "glmmTMB_vs_secondary_test_significant_results_by_gamma_band.csv"
full_path = Path(OUTPUT_DIR) / "glmmTMB_vs_secondary_test_concordance_by_gamma_band.csv"

if "glmmTMB_vs_secondary_test" in globals():
    sig_summary = glmmTMB_vs_secondary_test.loc[
        glmmTMB_vs_secondary_test["glmm_sig"] | glmmTMB_vs_secondary_test["secondary_sig"]
    ].copy()
elif summary_path.exists():
    sig_summary = pd.read_csv(summary_path)
elif full_path.exists():
    full = pd.read_csv(full_path)
    sig_summary = full.loc[full["glmm_sig"] | full["secondary_sig"]].copy()
else:
    raise FileNotFoundError(
        "Run the GLMM-TMB vs secondary-test concordance cell first."
    )

ordered_cols = [
    "metric", "environment", "gamma_band", "glmm_family",
    "glmm_estimate", "glmm_p", "glmm_sig_label",
    "secondary_test", "secondary_p", "secondary_sig_label", "secondary_statistic",
    "WT_value", "NLGF_value", "n_WT", "n_NLGF", "concordance",
]
ordered_cols = [col for col in ordered_cols if col in sig_summary.columns]

sig_summary = sig_summary.sort_values(["environment", "metric", "gamma_band"]).reset_index(drop=True)
sig_summary.to_csv(summary_path, index=False)

print(f"Significant GLMM-TMB or secondary-test rows: {len(sig_summary)}")
print(f"Saved: {summary_path}")

if sig_summary.empty:
    print(f"No significant GLMM-TMB or secondary-test genotype contrasts at alpha = {ALPHA}.")
else:
    display(HTML("<h3>Significant GLMM-TMB or secondary-test genotype effects</h3>"))
    display(
        sig_summary[ordered_cols].style
        .format({
            "glmm_estimate": "{:.6f}",
            "glmm_p": "{:.4f}",
            "secondary_p": "{:.4f}",
            "secondary_statistic": "{:.2f}",
            "WT_value": "{:.6f}",
            "NLGF_value": "{:.6f}",
        })
        .set_properties(**{"font-size": "11px"})
    )


## Two-way Linear Mixed-effects model per environmnet 
- PAC ~ Genotype * gamma_band + (1 | mouse_name) --> LMER (GAUSSIAN) - do not use


In [ ]:
# # ── 2-way mixed model per environment: PAC ~ Genotype * gamma_band + (1 | mouse_name) ────

# from rpy2.robjects import pandas2ri
# import pandas as pd
# import numpy as np
# from pathlib import Path
# import rpy2.robjects as ro
# from rpy2.robjects.conversion import localconverter

# ro.r("suppressPackageStartupMessages(library(lme4))")
# ro.r("suppressPackageStartupMessages(library(lmerTest))")
# ro.r("suppressPackageStartupMessages(library(emmeans))")

# SLOW_COL = 'comodulogram_PAC_slow_gamma'
# FAST_COL = 'comodulogram_PAC_fast_gamma'

# def sig_label(p):
#     if pd.isna(p): return 'NA'
#     if p < 0.001: return '***'
#     if p < 0.01:  return '**'
#     if p < 0.05:  return '*'
#     return 'ns'

# # ── Loop over each environment ─────────────────────────────────────────────────
# all_env_results = {}

# for env in sorted(df['environment'].dropna().unique()):
#     print(f"\n{'='*70}\nENVIRONMENT: {env}\n{'='*70}")
    
#     # Subset to this environment
#     sub_df = df[df['environment'] == env].copy()
    
#     # Build long-format PAC (slow + fast) for this environment
#     slow = sub_df[['mouse_name', 'Genotype', SLOW_COL]].rename(columns={SLOW_COL: 'PAC'})
#     slow['gamma_band'] = 'slow'
#     fast = sub_df[['mouse_name', 'Genotype', FAST_COL]].rename(columns={FAST_COL: 'PAC'})
#     fast['gamma_band'] = 'fast'
    
#     long_pac = pd.concat([slow, fast], ignore_index=True).dropna(subset=['PAC'])
#     long_pac['mouse_name']  = long_pac['mouse_name'].astype(str)
#     long_pac['Genotype']    = long_pac['Genotype'].astype(str)
#     long_pac['gamma_band']  = long_pac['gamma_band'].astype(str)
    
#     print(f"n rows = {len(long_pac)}, n mice = {long_pac['mouse_name'].nunique()}")
#     print(long_pac.groupby(['Genotype', 'gamma_band'])['PAC'].agg(['count', 'median']))
    
#     if long_pac['mouse_name'].nunique() < 4:
#         print("  Skipped: insufficient data")
#         continue
    
#     # ── Pass to R and fit 2-way model (no environment term) ─────────────────
#     with localconverter(ro.default_converter + pandas2ri.converter):
#         r_df = ro.conversion.py2rpy(long_pac)
#     ro.globalenv['dat'] = r_df
    
#     r_code = f"""
#     suppressWarnings({{
#         dat$mouse_name  <- factor(dat$mouse_name)
#         dat$Genotype    <- factor(dat$Genotype,    levels = c("WT", "NLGF"))
#         dat$gamma_band  <- factor(dat$gamma_band,  levels = c("slow", "fast"))
        
#         contrasts(dat$Genotype)   <- contr.sum(nlevels(dat$Genotype))
#         contrasts(dat$gamma_band) <- contr.sum(nlevels(dat$gamma_band))
        
#         # 2-way model: PAC ~ Genotype * gamma_band + (1 | mouse_name)
#         fit <- lmer(PAC ~ Genotype * gamma_band + (1 | mouse_name),
#                     data = dat, REML = FALSE)
        
#         # Type III ANOVA
#         aov_table <- as.data.frame(anova(fit, type = 3))
#         aov_table <- data.frame(term = rownames(aov_table), aov_table,
#                                 row.names = NULL, check.names = FALSE)
        
#         # 1. Genotype contrast within each gamma band
#         emm_geno_by_band <- emmeans(fit, ~ Genotype | gamma_band)
#         geno_by_band_df  <- as.data.frame(contrast(emm_geno_by_band, method = "pairwise"))
        
#         # 2. Gamma band contrast within each genotype
#         emm_band_by_geno <- emmeans(fit, ~ gamma_band | Genotype)
#         band_by_geno_df  <- as.data.frame(contrast(emm_band_by_geno, method = "pairwise"))
        
#         # 3. Interaction contrast
#         emm_full <- emmeans(fit, ~ Genotype * gamma_band)
#         interaction_df <- as.data.frame(contrast(emm_full, interaction = "pairwise"))
        
#         re_var  <- as.numeric(VarCorr(fit)$mouse_name[1,1])
#         aic_val <- AIC(fit)
        
#         list(
#             anova_table        = aov_table,
#             geno_by_band       = geno_by_band_df,
#             band_by_geno       = band_by_geno_df,
#             interaction        = interaction_df,
#             re_var            = re_var,
#             aic               = aic_val
#         )
#     }})
#     """
    
#     try:
#         out = ro.r(r_code)
        
#         with localconverter(ro.default_converter + pandas2ri.converter):
#             aov_df          = ro.conversion.rpy2py(out.rx2('anova_table'))
#             geno_by_band    = ro.conversion.rpy2py(out.rx2('geno_by_band'))
#             band_by_geno    = ro.conversion.rpy2py(out.rx2('band_by_geno'))
#             interaction_df  = ro.conversion.rpy2py(out.rx2('interaction'))
        
#         re_var  = float(out.rx2('re_var')[0])
#         aic_val = float(out.rx2('aic')[0])
        
#         print(f"\nAIC: {aic_val:.2f}  |  RE variance (mouse): {re_var:.6f}")
        
#         print("\n─── Type III ANOVA ───────────────────────────────────────")
#         print(aov_df.to_string(index=False))
        
#         print("\n─── Genotype (NLGF vs WT) within each gamma band ────────")
#         print(geno_by_band.to_string(index=False))
        
#         print("\n─── Gamma band (fast vs slow) within each genotype ──────")
#         print(band_by_geno.to_string(index=False))
        
#         print("\n─── Interaction contrast ──────────────────────────────────")
#         print(interaction_df.to_string(index=False))
        
#         # Store for later reference
#         all_env_results[env] = {
#             'anova': aov_df,
#             'geno_by_band': geno_by_band,
#             'band_by_geno': band_by_geno,
#             'interaction': interaction_df,
#         }
        
#         # Save results
#         env_clean = env.replace(" ", "_")
#         out_dir = Path(OUTPUT_DIR)
#         out_dir.mkdir(parents=True, exist_ok=True)
        
#         aov_df.to_csv(out_dir / f"PAC_lmer_2way_{env_clean}_anova.csv", index=False)
#         geno_by_band.to_csv(out_dir / f"PAC_lmer_2way_{env_clean}_geno_by_band.csv", index=False)
#         band_by_geno.to_csv(out_dir / f"PAC_lmer_2way_{env_clean}_band_by_geno.csv", index=False)
#         interaction_df.to_csv(out_dir / f"PAC_lmer_2way_{env_clean}_interaction.csv", index=False)
        
#         print(f"\n✅ Saved results for {env}")
        
#     except Exception as e:
#         print(f"FAILED: {e}")

# print(f"\n{'='*70}")
# print(f"✅ All environment-specific models completed and saved to: {OUTPUT_DIR}")

## Repeated measures ANOVA 

In [ ]:
# # ── Repeated-measures ANOVA (R via rpy2): gamma_band (within) × Genotype (between) ──
# import pandas as pd
# import numpy as np
# from pathlib import Path
# import rpy2.robjects as ro
# from rpy2.robjects import pandas2ri
# from rpy2.robjects.conversion import localconverter

# ro.r("suppressPackageStartupMessages(library(afex))")

# # ── Load data ──────────────────────────────────────────────────────────────
# OUTPUT_DIR = LFP_CSV_PATH.parent / 'PAC_rm_anova_R_by_env'
# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# df = pd.read_csv(LFP_CSV_PATH)
# df['mouse_name']  = df['mouse_name'].astype(str)
# df['Genotype']    = df['Genotype'].astype(str)
# df['environment'] = df['environment'].astype(str)

# SLOW_COL = 'comodulogram_PAC_slow_gamma'
# FAST_COL = 'comodulogram_PAC_fast_gamma'

# def sig_label(p):
#     if pd.isna(p): return 'NA'
#     if p < 0.001: return '***'
#     if p < 0.01:  return '**'
#     if p < 0.05:  return '*'
#     return 'ns'

# def build_long_pac(sub_env):
#     slow = sub_env[['mouse_name', 'Genotype', SLOW_COL]].rename(columns={SLOW_COL: 'PAC'})
#     slow['gamma_band'] = 'slow'
#     fast = sub_env[['mouse_name', 'Genotype', FAST_COL]].rename(columns={FAST_COL: 'PAC'})
#     fast['gamma_band'] = 'fast'
#     return pd.concat([slow, fast], ignore_index=True).dropna(subset=['PAC'])

# def run_rm_anova_R(long_df):
#     """Fit PAC ~ Genotype * gamma_band with mouse_name as within-subject error term, via afex::aov_ez."""
#     with localconverter(ro.default_converter + pandas2ri.converter):
#         r_df = ro.conversion.py2rpy(long_df)
#     ro.globalenv['dat'] = r_df

#     r_code = """
#     suppressWarnings({
#         dat$mouse_name <- factor(dat$mouse_name)
#         dat$Genotype   <- factor(dat$Genotype, levels = c("WT", "NLGF"))
#         dat$gamma_band <- factor(dat$gamma_band, levels = c("slow", "fast"))
#         dat <- droplevels(dat)

#         fit <- aov_ez(
#             id      = "mouse_name",
#             dv      = "PAC",
#             data    = dat,
#             between = "Genotype",
#             within  = "gamma_band",
#             type    = 3,
#             anova_table = list(correction = "GG")   # Greenhouse-Geisser, moot here w/ 2 levels but safe default
#         )

#         aov_df <- as.data.frame(fit$anova_table)
#         aov_df <- data.frame(term = rownames(aov_df), aov_df, row.names = NULL, check.names = FALSE)

#         list(anova_table = aov_df)
#     })
#     """
#     out = ro.r(r_code)
#     with localconverter(ro.default_converter + pandas2ri.converter):
#         aov_df = ro.conversion.rpy2py(out.rx2('anova_table'))
#     return aov_df

# # ── Run separately per environment ───────────────────────────────────────────
# all_results = {}
# rm_rows_summary = []

# for env in sorted(df['environment'].dropna().unique()):
#     print(f"\n{'='*70}\nENVIRONMENT: {env}\n{'='*70}")

#     sub_env  = df[df['environment'] == env].copy()
#     long_pac = build_long_pac(sub_env)

#     # ── Check balance: every mouse must have exactly 2 rows (slow + fast) ────
#     counts = long_pac.groupby('mouse_name').size()
#     unbalanced = counts[counts != 2]
#     if len(unbalanced) > 0:
#         print(f"⚠️  Unbalanced design — these mice don't have both gamma bands, dropping them:")
#         print(unbalanced)
#         long_pac = long_pac[~long_pac['mouse_name'].isin(unbalanced.index)]

#     n_mice = long_pac['mouse_name'].nunique()
#     print(f"n rows = {len(long_pac)}, n mice (balanced) = {n_mice}, "
#           f"genotypes = {sorted(long_pac['Genotype'].unique())}")

#     if n_mice < 4:
#         print("  Skipped: insufficient balanced data")
#         continue

#     try:
#         aov_df = run_rm_anova_R(long_pac)
#         print("\nRepeated-measures ANOVA (afex::aov_ez, Type III):")
#         print(aov_df.to_string(index=False))

#         all_results[env] = aov_df

#         # afex column names: 'num Df', 'den Df', 'MSE', 'F', 'ges', 'Pr(>F)'
#         p_col = [c for c in aov_df.columns if 'Pr' in c][0]
#         for _, row in aov_df.iterrows():
#             rm_rows_summary.append({
#                 'environment': env,
#                 'term':        row['term'],
#                 'F':           row.get('F', np.nan),
#                 'p_value':     row.get(p_col, np.nan),
#                 'sig':         sig_label(row.get(p_col, np.nan)),
#             })

#         aov_df.to_csv(OUTPUT_DIR / f"PAC_rm_anova_R_{env}.csv", index=False)
#         print(f"Saved: {OUTPUT_DIR / f'PAC_rm_anova_R_{env}.csv'}")

#     except Exception as e:
#         print(f"  FAILED: {e}")

# df_summary = pd.DataFrame(rm_rows_summary)
# summary_path = OUTPUT_DIR / "PAC_rm_anova_R_summary_both_envs.csv"
# df_summary.to_csv(summary_path, index=False)
# print(f"\n✅ Combined summary saved to: {summary_path}")
# print(df_summary.to_string(index=False))